<h2>Generate audio from text</h2>

In [ ]:
import os
# Add these BEFORE any imports
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
from dotenv import load_dotenv

import requests
import time
import csv
from pydub import AudioSegment
from transformers import pipeline

import scipy
import numpy as np
from transformers import AutoProcessor, BarkModel

load_dotenv(override=True)
HF_TOKEN = os.getenv("HUGGING_FACE_AUDIO_TOKEN")

import torch

from huggingface_hub import login
#Authenticates you with Hugging Face Hub (e.g., to access private models, datasets, or paid APIs).
login(token=HF_TOKEN)

In [25]:
# Load model (downloads once, ~5GB, cached after)
print("⏳ Loading Bark model (first time takes a few minutes)...")
# Force CPU explicitly
torch.backends.mps.is_available = lambda: False  # disable MPS entirely

# synthesiser = pipeline(     #may load from cache "~/.cache/huggingface/hub"
#     "text-to-speech",
#     "suno/bark-small",
#     device="cpu"
# )
processor = AutoProcessor.from_pretrained("suno/bark-small")
model = BarkModel.from_pretrained("suno/bark-small").to("cpu")
print("✅ Model loaded!")

⏳ Loading Bark model (first time takes a few minutes)...


Loading weights: 100%|██████████| 542/542 [00:00<00:00, 65906.50it/s]
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.1.weight to fine_acoustics.lm_heads.0.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.2.weight to fine_acoustics.lm_heads.1.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.3.weight to fine_acoustics.lm_heads.2.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping an

✅ Model loaded!


In [28]:
# def generate_audio_2_file(text, output_filepath):

# Pick a voice preset
# Options: v2/en_speaker_0 to v2/en_speaker_9
VOICE_PRESET = "v2/en_speaker_6"  # natural male voice

def generate_audio(text, output_filename):
    inputs = processor(text, voice_preset=VOICE_PRESET, return_tensors="pt")
    
    # Make sure all tensors are on CPU
    inputs = {k: v.to("cpu") for k, v in inputs.items()}

    with torch.no_grad():
        audio_array = model.generate(**inputs)

    audio_array = audio_array.cpu().numpy().squeeze()
    sample_rate = model.generation_config.sample_rate

    # Save temp WAV then convert to MP3
    temp_wav = output_filename.replace(".mp3", "_temp.wav")
    audio_array = audio_array / np.max(np.abs(audio_array))
    audio_int16 = (audio_array * 32767).astype(np.int16)
    scipy.io.wavfile.write(temp_wav, rate=sample_rate, data=audio_int16)

    audio = AudioSegment.from_wav(temp_wav)
    audio.export(output_filename, format="mp3", bitrate="192k")
    os.remove(temp_wav)

    print(f"✅ Saved: {output_filename}")


# ---- BATCH FROM CSV ----
def batch_generate(csv_file, output_folder="audio_output"):
    os.makedirs(output_folder, exist_ok=True)

    with open(csv_file, "r") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            output_path = os.path.join(output_folder, row["file"])
            print(f"\n🎙️ Generating {i+1}: {row['file']}")
            generate_audio(row["text"], output_path)

    print("\n🎉 All done!")


# Test single audio first
# generate_audio(
#     "Welcome back to the channel! Today we have something amazing for you.",
#     "test_intro.mp3"
# )